# FCFF DCF Valuation v1
## Sales-driven FCFF 모델 기반 미국 주식 내재가치 계산

### 처리 흐름
```
①  DB (us_revenue_forecast_data) → 과거 Sales + 12분기 예측 Sales
②  FMP API → IS / BS / CF 분기 데이터
③  Sales-driven 회귀
      OPM  = f(Sales)      [SARIMA/ETS/Theta Ensemble 예측]
      D&A  = α × Sales     [OLS 또는 median ratio fallback]
      CapEx= β × Sales     [OLS 또는 median ratio fallback]
      NWC  = γ × Sales     [OLS 또는 median ratio fallback]
      ΔNWC = NWC_t - NWC_{t-1}
④  FCFF = EBIT×(1-tax) + D&A - CapEx - ΔNWC
⑤  WACC = Re×(E/V) + Rd×(1-tax)×(D/V)
      Re   : us_rim_spread_data 의 Re_smooth
⑥  Terminal Value = FCFF_last×(1+g) / (WACC-g)
⑦  EV = Σ FCFF/(1+WACC)^t + TV
      Target Price = (EV - Net Debt) / shares
⑧  DB 저장 + 시각화
```

---
| 셀  | 역할 |
|-----|------|
| 1   | 경로 자동 감지 |
| 2   | Import & 설정 상수 |
| 3   | DB 연결 & 테이블 초기화 |
| 4   | DCFModel 클래스 정의 |
| 5   | 배치 실행 |
| 6   | 결과 조회 & 시각화 |


## Cell 1 · 경로 자동 감지

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate
    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다.\n"
        "_CANDIDATE_ROOTS 를 현재 환경에 맞게 수정하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")
print(f"[확인] DATA 경로    : {os.path.join(_ROOT, 'DATA')}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로    : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


## Cell 2 · Import & 설정 상수

> **여기만 수정하면 됩니다**: `TICKER_START / TICKER_END / SKIP_DONE / DISCOUNT_MODE`

In [2]:
# ── 표준 라이브러리 ──────────────────────────────────────────────
import gc, math, time, traceback, warnings
import requests
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple

# ── 외부 라이브러리 ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import text

# ── 내부 모듈 ────────────────────────────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as US_TICKER_LIST
from DATA.universal_ts_forecast_function_v2 import (
    forecast_sarima, forecast_ets, forecast_theta,
    infer_freq_alias, seasonal_periods_from_freq, clear_memory,
)

def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ══════════════════════════════════════════════════════════════════
#  설정 상수 — 여기만 수정하세요
# ══════════════════════════════════════════════════════════════════
FMP_API_KEY   = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
FMP_BASE      = "https://financialmodelingprep.com/api/v3"
FMP_LIMIT     = 40          # 분기 수 (최대 조회)
FMP_SLEEP     = 0.35        # API 호출 간격 (초)

# DB 테이블
TABLE_SALES    = "us_revenue_forecast_data"   # Sales 예측 결과
TABLE_RE       = "us_rim_spread_data"         # Re_smooth
TABLE_RESULT   = "us_fcff_dcf_valuation"      # 저장 결과
DEFAULT_PORT   = 3307

# 모델 파라미터
FORECAST_HORIZON   = 12        # 예측 분기 수
MIN_HISTORY        = 16        # OPM 회귀 최소 분기 수
WINSORIZE_LIMITS   = (0.05, 0.95)
DISCOUNT_MODE      = "wacc"    # 'wacc' or 're'
GDP_GROWTH         = 0.04      # terminal growth 상한

# OLS fallback 기준
OLS_MIN_R2         = 0.30
OLS_MIN_SAMPLES    = 12

# 배치 범위 (필요 시 조정)
TICKER_START = 0
TICKER_END   = 10 #len(US_TICKER_LIST)
SKIP_DONE    = False

CHECKPOINT_DIR = "_fcff_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")
FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")

db_info = get_db_info()
engine  = get_engine(db_info)

print(f"[OK] Import 완료  |  대상 티커: {len(US_TICKER_LIST):,}개")
print(f"[설정] DISCOUNT_MODE={DISCOUNT_MODE}  FORECAST_HORIZON={FORECAST_HORIZON}  GDP_GROWTH={GDP_GROWTH}")


[OK] Import 완료  |  대상 티커: 2,000개
[설정] DISCOUNT_MODE=wacc  FORECAST_HORIZON=12  GDP_GROWTH=0.04


## Cell 3 · DB 연결 & 결과 테이블 초기화

In [3]:
# ── DB 연결 함수 ────────────────────────────────────────────────
def get_conn():
    return pymysql.connect(
        host       = db_info["host"],
        port       = int(db_info.get("port", DEFAULT_PORT)),
        user       = db_info["user"],
        password   = db_info["password"],
        db         = db_info.get("database", "investar"),
        charset    = "utf8mb4",
        autocommit = False,
        cursorclass= pymysql.cursors.DictCursor,
    )

# ── 결과 테이블 생성 ─────────────────────────────────────────────
CREATE_SQL = """
CREATE TABLE IF NOT EXISTS `us_fcff_dcf_valuation` (
  `id`                BIGINT      NOT NULL AUTO_INCREMENT,
  `date`              DATE        NOT NULL COMMENT '예측 실행일',
  `ticker`            VARCHAR(20) NOT NULL,
  `quarter`           VARCHAR(10)          COMMENT 'e.g. 2026Q1',
  `sales_actual`      DOUBLE               COMMENT '과거 Sales',
  `sales_forecast`    DOUBLE               COMMENT '예측 Sales',
  `opm_forecast`      DOUBLE               COMMENT '예측 OPM',
  `ebit`              DOUBLE,
  `tax_rate`          DOUBLE,
  `nopat`             DOUBLE,
  `da`                DOUBLE,
  `capex`             DOUBLE,
  `nwc`               DOUBLE,
  `delta_nwc`         DOUBLE,
  `fcff`              DOUBLE,
  `roic`              DOUBLE,
  `reinvestment_rate` DOUBLE,
  `g_terminal`        DOUBLE,
  `discount_rate`     DOUBLE,
  `enterprise_value`  DOUBLE,
  `net_debt`          DOUBLE,
  `equity_value`      DOUBLE,
  `shares`            DOUBLE,
  `target_price`      DOUBLE,
  `current_price`     DOUBLE,
  `upside_pct`        DOUBLE,
  `created_at`        DATETIME    DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY (`id`),
  UNIQUE KEY uq_main (`ticker`, `date`, `quarter`),
  INDEX idx_ticker (`ticker`),
  INDEX idx_date   (`date`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""

conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_SQL)
    conn.commit()
    log("DB", f"테이블 준비 완료: {TABLE_RESULT}")
finally:
    conn.close()

# 연결 테스트
try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공  host={db_info.get('host')}  port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")


[23:14:03][DB] 테이블 준비 완료: us_fcff_dcf_valuation
[23:14:03][DB] 연결 성공  host=192.168.0.230  port=3307


## Cell 4 · DCFModel 클래스

### 클래스 구조
| 메서드 | 역할 |
|--------|------|
| `load_sales()` | DB → 과거 + 예측 Sales |
| `load_financials()` | FMP → IS/BS/CF |
| `estimate_tax_rate()` | 실효세율 (pretaxIncome>0 분기) |
| `estimate_opm()` | SARIMA/ETS/Theta Ensemble OPM 예측 |
| `estimate_ratio_coef()` | D&A/CapEx OLS 또는 median ratio |
| `estimate_nwc_coef()` | NWC/Sales 계수 |
| `compute_fcff()` | FCFF = NOPAT + D&A - CapEx - ΔNWC |
| `compute_wacc()` | WACC (Re_smooth + Rd) |
| `compute_terminal_growth()` | g = min(GDP, 0.5×CAGR + 0.5×ROIC×reinv) |
| `compute_valuation()` | DCF → EV → TP |
| `save_to_db()` | 결과 DB 저장 |
| `plot()` | 4패널 시각화 |
| `run()` | 전체 실행 |

In [4]:
# ═══════════════════════════════════════════════════════════════
#  DCFModel 클래스
# ═══════════════════════════════════════════════════════════════

class DCFModel:
    """
    Sales-driven FCFF DCF Valuation Model

    Parameters
    ----------
    ticker : str
    engine : SQLAlchemy engine  (Sales DB 조회용)
    fmp_api_key : str
    forecast_horizon : int      (예측 분기 수, default 12)
    min_history : int           (OPM 회귀 최소 관측치, default 16)
    discount_mode : str         'wacc' or 're'
    gdp_growth : float          terminal growth 상한 (default 0.04)
    verbose : bool              디버그 출력 여부
    """

    def __init__(
        self,
        ticker: str,
        engine,
        fmp_api_key: str = FMP_API_KEY,
        forecast_horizon: int = FORECAST_HORIZON,
        min_history: int = MIN_HISTORY,
        discount_mode: str = DISCOUNT_MODE,
        gdp_growth: float = GDP_GROWTH,
        verbose: bool = False,
    ):
        self.ticker           = ticker.upper()
        self.engine           = engine
        self.api_key          = fmp_api_key
        self.horizon          = forecast_horizon
        self.min_history      = min_history
        self.discount_mode    = discount_mode
        self.gdp_growth       = gdp_growth
        self.verbose          = verbose

        # 내부 저장소
        self._sales_actual:   Optional[pd.Series] = None
        self._sales_forecast: Optional[pd.Series] = None
        self._inc:            Optional[pd.DataFrame] = None
        self._bs:             Optional[pd.DataFrame] = None
        self._cf:             Optional[pd.DataFrame] = None
        self._fcff_history:   Optional[pd.Series] = None
        self.result_df:       Optional[pd.DataFrame] = None
        self.valuation:       Optional[Dict] = None

    # ─────────────────────────────────────────────────────────────
    # 1. 데이터 로드
    # ─────────────────────────────────────────────────────────────

    def load_sales(self) -> "DCFModel":
        """DB에서 과거 Sales(actual) + 예측 Sales(forecast Ensemble) 로드"""
        sql = text(f"""
            SELECT date, data_type, model, value
            FROM   `{TABLE_SALES}`
            WHERE  ticker = :t AND item = 'sale'
            ORDER  BY date
        """)
        with self.engine.connect() as conn:
            df = pd.read_sql(sql, conn, params={"t": self.ticker})

        if df.empty:
            raise ValueError(f"[{self.ticker}] Sales DB 데이터 없음 — us_revenue_forecast_notebook 먼저 실행")

        df["date"]  = pd.to_datetime(df["date"])
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

        actual = (df[df["data_type"] == "actual"]
                  .sort_values("date")
                  .drop_duplicates("date", keep="last")
                  .set_index("date")["value"])

        # Ensemble 우선 → 없으면 SARIMA → 없으면 ETS
        for model_name in ["Ensemble", "SARIMA", "ETS", "Theta"]:
            fc = (df[(df["data_type"] == "forecast") & (df["model"] == model_name)]
                  .sort_values("date")
                  .drop_duplicates("date", keep="last")
                  .set_index("date")["value"])
            if len(fc) >= self.horizon:
                forecast = fc.iloc[:self.horizon]
                break
        else:
            raise ValueError(f"[{self.ticker}] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)")

        self._sales_actual   = actual
        self._sales_forecast = forecast
        if self.verbose:
            log(self.ticker, f"Sales actual={len(actual)}분기 / forecast={len(forecast)}분기")
        return self

    def _fmp_get(self, endpoint: str, period: str = "quarter", limit: int = FMP_LIMIT) -> pd.DataFrame:
        """FMP API GET → DataFrame"""
        url = f"{FMP_BASE}/{endpoint}/{self.ticker}"
        for attempt in range(3):
            try:
                r = requests.get(url, params={"period": period, "limit": limit, "apikey": self.api_key}, timeout=20)
                if r.status_code == 429:
                    time.sleep(1.5 + attempt); continue
                r.raise_for_status()
                data = r.json()
                if isinstance(data, dict) and "Error Message" in data:
                    return pd.DataFrame()
                return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame()
            except Exception as e:
                if attempt == 2:
                    if self.verbose: log(self.ticker, f"FMP {endpoint} 실패: {e}")
                    return pd.DataFrame()
                time.sleep(FMP_SLEEP + attempt * 0.5)
        return pd.DataFrame()

    def _parse_date_col(self, df: pd.DataFrame) -> pd.DataFrame:
        """FMP date 컬럼 파싱 + 분기말 날짜로 통일 (acceptedDate 기반 look-ahead bias 보정)"""
        df = df.copy()
        df["report_date"] = pd.to_datetime(
            df.get("acceptedDate", df.get("fillingDate", pd.NaT)), errors="coerce"
        )
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        # look-ahead bias 보정: acceptedDate 있으면 report_date - 45일 → 회계분기말
        def _qend(row):
            if pd.notna(row.get("report_date")):
                return (row["report_date"] - pd.Timedelta(days=45)).to_period("Q").to_timestamp("Q")
            return row["date"].to_period("Q").to_timestamp("Q")
        df["date"] = df.apply(_qend, axis=1)
        df = df.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)
        return df

    def load_financials(self) -> "DCFModel":
        """FMP API → IS / BS / CF 분기 데이터 로드"""
        time.sleep(FMP_SLEEP)
        inc = self._fmp_get("income-statement")
        time.sleep(FMP_SLEEP)
        bs  = self._fmp_get("balance-sheet-statement")
        time.sleep(FMP_SLEEP)
        cf  = self._fmp_get("cash-flow-statement")

        if inc.empty or bs.empty or cf.empty:
            raise ValueError(f"[{self.ticker}] FMP 재무제표 로드 실패 (IS/BS/CF 중 하나 이상 비어있음)")

        self._inc = self._parse_date_col(inc)
        self._bs  = self._parse_date_col(bs)
        self._cf  = self._parse_date_col(cf)
        if self.verbose:
            log(self.ticker, f"IS={len(self._inc)}Q / BS={len(self._bs)}Q / CF={len(self._cf)}Q")
        return self

    # ─────────────────────────────────────────────────────────────
    # 2. 변수별 추정
    # ─────────────────────────────────────────────────────────────

    @staticmethod
    def _winsorize(s: pd.Series, limits=WINSORIZE_LIMITS) -> pd.Series:
        lo, hi = s.quantile(limits[0]), s.quantile(limits[1])
        return s.clip(lo, hi)

    @staticmethod
    def _ols_ratio(x: pd.Series, y: pd.Series) -> Tuple[float, float]:
        """y = slope*x 단순회귀. R² 반환"""
        mask = x.notna() & y.notna() & (x != 0)
        if mask.sum() < OLS_MIN_SAMPLES:
            return np.nan, -1.0
        slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
        return slope, r ** 2

    def estimate_tax_rate(self) -> float:
        """실효세율 추정 (pretaxIncome > 0 분기만 사용, 범위 0~40%)"""
        inc = self._inc
        cols_needed = ["pretaxIncome", "incomeTaxExpense"]
        if not all(c in inc.columns for c in cols_needed):
            return 0.21  # 법정세율 fallback
        df = inc[cols_needed].apply(pd.to_numeric, errors="coerce")
        df = df[df["pretaxIncome"] > 0].dropna()
        if df.empty:
            return 0.21
        rates = (df["incomeTaxExpense"] / df["pretaxIncome"]).clip(0, 0.40)
        return float(rates.median())

    def estimate_opm(self, sales_series: pd.Series) -> pd.Series:
        """
        OPM 예측: SARIMA/ETS/Theta Ensemble
        fallback: best single → 최근 8분기 median → 전체 median
        과거 IS에서 OPM = operatingIncome / revenue 계산 후 예측
        """
        inc = self._inc
        if "operatingIncome" not in inc.columns or "revenue" not in inc.columns:
            fallback_opm = 0.10
            if self.verbose: log(self.ticker, "OPM: IS 컬럼 없음 → 10% fallback")
            return pd.Series([fallback_opm] * len(sales_series), index=sales_series.index)

        df = inc[["date","operatingIncome","revenue"]].copy()
        df["operatingIncome"] = pd.to_numeric(df["operatingIncome"], errors="coerce")
        df["revenue"]         = pd.to_numeric(df["revenue"], errors="coerce")
        df = df[(df["revenue"] > 0)].dropna().set_index("date").sort_index()
        df["opm"] = df["operatingIncome"] / df["revenue"]
        df["opm"] = self._winsorize(df["opm"])

        opm_series = df["opm"]
        if len(opm_series) < self.min_history:
            fallback = float(opm_series.median()) if not opm_series.empty else 0.10
            if self.verbose: log(self.ticker, f"OPM: 데이터 부족({len(opm_series)}) → median={fallback:.4f}")
            return pd.Series([fallback] * len(sales_series), index=sales_series.index)

        freq_alias = infer_freq_alias(opm_series.index)
        m = seasonal_periods_from_freq(freq_alias)

        forecasts = {}
        for name, fn in [("SARIMA", lambda y: forecast_sarima(y, self.horizon, seasonal_period=m)),
                         ("ETS",    lambda y: forecast_ets(y, self.horizon, m=m)),
                         ("Theta",  lambda y: forecast_theta(y, self.horizon, m=m))]:
            try:
                res = fn(opm_series)
                if "forecast" in res and "error" not in res:
                    forecasts[name] = np.asarray(res["forecast"])
            except Exception:
                pass

        if forecasts:
            ens = np.mean(list(forecasts.values()), axis=0)
            ens = np.clip(ens, -0.50, 0.80)  # OPM 범위 안정화
            return pd.Series(ens, index=sales_series.index)

        # Fallback: 최근 8분기 median → 전체 median
        fallback = float(opm_series.tail(8).median() if len(opm_series) >= 8
                         else opm_series.median())
        fallback = np.clip(fallback, -0.50, 0.80)
        if self.verbose: log(self.ticker, f"OPM: 예측 실패 → median fallback={fallback:.4f}")
        return pd.Series([fallback] * len(sales_series), index=sales_series.index)

    def estimate_ratio_coef(
        self, cf_col: str, ratio_name: str, take_abs: bool = False
    ) -> Tuple[float, str]:
        """
        D&A / CapEx / NWC 비율 계수 추정.
        OLS slope 우선 → R²<0.30 or 표본부족 or 음수 slope → median ratio fallback
        반환: (coef, method_used)
        """
        # Sales 매칭
        inc = self._inc[["date","revenue"]].copy()
        inc["revenue"] = pd.to_numeric(inc["revenue"], errors="coerce")

        # 데이터 소스 선택 (IS/BS/CF)
        if cf_col in self._cf.columns:
            src = self._cf[["date", cf_col]].copy()
        elif cf_col in self._inc.columns:
            src = self._inc[["date", cf_col]].copy()
        elif cf_col in self._bs.columns:
            src = self._bs[["date", cf_col]].copy()
        else:
            if self.verbose: log(self.ticker, f"{ratio_name}: 컬럼 없음 → 0 반환")
            return 0.0, "missing"

        src[cf_col] = pd.to_numeric(src[cf_col], errors="coerce")
        if take_abs:
            src[cf_col] = src[cf_col].abs()

        merged = inc.merge(src, on="date", how="inner").dropna()
        merged = merged[merged["revenue"] > 0]

        if merged.empty:
            return 0.0, "empty"

        x = merged["revenue"]
        y = merged[cf_col]

        slope, r2 = self._ols_ratio(x, y)

        use_ols = (
            not np.isnan(slope)
            and r2 >= OLS_MIN_R2
            and slope >= 0
            and len(merged) >= OLS_MIN_SAMPLES
        )

        if use_ols:
            if self.verbose: log(self.ticker, f"{ratio_name}: OLS slope={slope:.6f} R²={r2:.3f}")
            return float(slope), "ols"
        else:
            ratios = (y / x).replace([np.inf, -np.inf], np.nan).dropna()
            if ratios.empty:
                return 0.0, "ratio_empty"
            ratios = self._winsorize(ratios)
            med = float(ratios.median())
            if self.verbose: log(self.ticker, f"{ratio_name}: OLS 부적합(R²={r2:.3f}) → median ratio={med:.6f}")
            return max(med, 0.0), "median_ratio"

    def estimate_nwc_coef(self) -> Tuple[float, str]:
        """
        NWC = (Current Assets - Cash) - (Current Liabilities - Short-term Debt)
        NWC/Sales 비율 OLS or median
        """
        bs = self._bs.copy()
        needed = ["currentAssets", "cashAndCashEquivalents",
                  "totalCurrentLiabilities", "shortTermDebt"]
        for c in needed:
            if c not in bs.columns:
                bs[c] = 0.0
        for c in needed:
            bs[c] = pd.to_numeric(bs[c], errors="coerce").fillna(0)

        bs["nwc"] = (
            (bs["currentAssets"] - bs["cashAndCashEquivalents"])
            - (bs["totalCurrentLiabilities"] - bs["shortTermDebt"])
        )

        inc = self._inc[["date","revenue"]].copy()
        inc["revenue"] = pd.to_numeric(inc["revenue"], errors="coerce")
        merged = inc.merge(bs[["date","nwc"]], on="date", how="inner").dropna()
        merged = merged[merged["revenue"] > 0]

        if merged.empty:
            return 0.0, "empty"

        slope, r2 = self._ols_ratio(merged["revenue"], merged["nwc"])
        use_ols = (not np.isnan(slope) and r2 >= OLS_MIN_R2
                   and len(merged) >= OLS_MIN_SAMPLES)

        if use_ols:
            return float(slope), "ols"
        else:
            ratios = (merged["nwc"] / merged["revenue"]).replace([np.inf,-np.inf], np.nan).dropna()
            if ratios.empty:
                return 0.0, "ratio_empty"
            return float(self._winsorize(ratios).median()), "median_ratio"

    # ─────────────────────────────────────────────────────────────
    # 3. FCFF 계산
    # ─────────────────────────────────────────────────────────────

    def compute_fcff(self) -> "DCFModel":
        """Sales-driven FCFF = NOPAT + D&A - CapEx - ΔNWC"""
        sales_fc = self._sales_forecast
        tax      = self.estimate_tax_rate()
        opm_fc   = self.estimate_opm(sales_fc)

        alpha, _  = self.estimate_ratio_coef("depreciationAndAmortization", "D&A")
        beta, _   = self.estimate_ratio_coef("capitalExpenditure", "CapEx", take_abs=True)
        gamma, _  = self.estimate_nwc_coef()

        rows = []
        prev_nwc = None
        for i, (dt, sales) in enumerate(sales_fc.items()):
            opm    = float(opm_fc.iloc[i])
            ebit   = sales * opm
            nopat  = ebit * (1 - tax)
            da     = alpha * sales
            capex  = beta  * sales
            nwc    = gamma * sales
            delta_nwc = (nwc - prev_nwc) if prev_nwc is not None else 0.0
            prev_nwc = nwc

            fcff   = nopat + da - capex - delta_nwc
            # ROIC = NOPAT / (CapEx + ΔNWC)  (간략 추정)
            invested = capex + max(delta_nwc, 0)
            roic = nopat / invested if invested > 1e-6 else np.nan
            reinv = capex / nopat if nopat > 1e-6 else np.nan

            rows.append({
                "date":         dt,
                "quarter":      f"{dt.year}Q{dt.quarter}",
                "sales_forecast": sales,
                "opm_forecast": opm,
                "ebit":         ebit,
                "tax_rate":     tax,
                "nopat":        nopat,
                "da":           da,
                "capex":        capex,
                "nwc":          nwc,
                "delta_nwc":    delta_nwc,
                "fcff":         fcff,
                "roic":         roic,
                "reinvestment_rate": reinv,
            })

        self.result_df = pd.DataFrame(rows)

        # 역사적 FCFF (terminal growth CAGR 계산용)
        self._compute_historical_fcff(tax, alpha, beta, gamma)
        return self

    def _compute_historical_fcff(self, tax, alpha, beta, gamma):
        """과거 FCFF 시계열 계산 (terminal growth CAGR용)"""
        sales_act = self._sales_actual
        if sales_act is None or sales_act.empty:
            self._fcff_history = pd.Series(dtype=float)
            return

        inc = self._inc.set_index("date")
        rows = []
        prev_nwc = None
        for dt, sales in sales_act.items():
            opm_row = inc["operatingIncome"].get(dt, np.nan) if "operatingIncome" in inc.columns else np.nan
            rev_row = inc["revenue"].get(dt, np.nan) if "revenue" in inc.columns else np.nan
            opm = (float(opm_row) / float(rev_row)) if (pd.notna(opm_row) and pd.notna(rev_row) and float(rev_row) > 0) else np.nan
            if pd.isna(opm):
                prev_nwc = gamma * sales; continue
            ebit  = sales * opm
            nopat = ebit * (1 - tax)
            da    = alpha * sales
            capex = beta  * sales
            nwc   = gamma * sales
            dnwc  = (nwc - prev_nwc) if prev_nwc is not None else 0.0
            prev_nwc = nwc
            rows.append({"date": dt, "fcff": nopat + da - capex - dnwc})

        if rows:
            df_h = pd.DataFrame(rows).set_index("date")["fcff"]
            self._fcff_history = df_h
        else:
            self._fcff_history = pd.Series(dtype=float)

    # ─────────────────────────────────────────────────────────────
    # 4. WACC 계산
    # ─────────────────────────────────────────────────────────────

    def _get_re_smooth(self) -> Optional[float]:
        """DB us_rim_spread_data 에서 최신 Re_smooth 조회"""
        sql = text(f"""
            SELECT value FROM `{TABLE_RE}`
            WHERE ticker=:t AND indicator='Re_smooth'
            ORDER BY date DESC LIMIT 1
        """)
        try:
            with self.engine.connect() as conn:
                row = conn.execute(sql, {"t": self.ticker}).fetchone()
            if row:
                return float(row[0])
        except Exception:
            pass
        return None

    def compute_wacc(self) -> float:
        """WACC = Re×E/V + Rd×(1-tax)×D/V"""
        tax = self.estimate_tax_rate()
        re  = self._get_re_smooth()
        if re is None:
            # fallback: CAPM 대략값 (rf 4.5% + β1.0 × ERP 5.5%)
            re = 0.045 + 1.0 * 0.055
            if self.verbose: log(self.ticker, f"Re_smooth 없음 → CAPM fallback Re={re:.4f}")

        if self.discount_mode == "re":
            if self.verbose: log(self.ticker, f"discount_mode=re → WACC={re:.4f}")
            return float(re)

        # 부채 비용
        bs  = self._bs
        inc = self._inc
        rd  = 0.05  # 기본값

        if "totalDebt" in bs.columns and "interestExpense" in inc.columns:
            latest_bs  = bs.sort_values("date").iloc[-1]
            total_debt = pd.to_numeric(latest_bs.get("totalDebt", 0), errors="coerce") or 0

            ie = pd.to_numeric(inc["interestExpense"], errors="coerce").abs()
            total_debt_avg = pd.to_numeric(bs["totalDebt"], errors="coerce")
            total_debt_avg = (total_debt_avg + total_debt_avg.shift(1)) / 2

            valid = (total_debt_avg > 0) & ie.notna()
            if valid.sum() >= 2:
                rd_series = (ie[valid] / total_debt_avg[valid]).clip(0, 0.15)
                rd = float(rd_series.median())

            # 시가총액 (최신)
            mkt_cap = 0.0
            try:
                r = requests.get(f"{FMP_BASE}/market-capitalization/{self.ticker}",
                                  params={"apikey": self.api_key}, timeout=10)
                data = r.json()
                if isinstance(data, list) and data:
                    mkt_cap = float(data[0].get("marketCap", 0) or 0)
            except Exception:
                pass

            V = mkt_cap + total_debt
            if V > 1e6:
                we = mkt_cap / V
                wd = total_debt / V
                wacc = re * we + rd * (1 - tax) * wd
                wacc = float(np.clip(wacc, 0.04, 0.25))
                if self.verbose: log(self.ticker, f"WACC={wacc:.4f}  Re={re:.4f} Rd={rd:.4f} tax={tax:.4f}")
                return wacc

        if self.verbose: log(self.ticker, f"WACC 계산 불완전 → Re fallback={re:.4f}")
        return float(re)

    # ─────────────────────────────────────────────────────────────
    # 5. Terminal Growth
    # ─────────────────────────────────────────────────────────────

    def compute_terminal_growth(self, reinvestment_rate: float, wacc: float) -> float:
        """
        g = min(GDP_growth, 0.5*FCFF_CAGR + 0.5*(ROIC*reinv_rate))
        g ∈ [0, GDP_growth]
        """
        # FCFF CAGR (최근 20분기, 시작/끝 양수 조건)
        fcff_cagr = 0.0
        h = self._fcff_history
        if h is not None and len(h) >= 4:
            h20 = h.dropna().tail(20)
            f0, fl = float(h20.iloc[0]), float(h20.iloc[-1])
            n = len(h20)
            if f0 > 0 and fl > 0 and n >= 4:
                fcff_cagr = (fl / f0) ** (4.0 / n) - 1.0  # 연환산 (4분기)
                fcff_cagr = float(np.clip(fcff_cagr, -0.20, 0.40))

        # ROIC 기반 성장
        df_hist = self.result_df
        roic_vals = df_hist["roic"].dropna()
        roic_med  = float(roic_vals.median()) if not roic_vals.empty else 0.08
        g_roic = roic_med * reinvestment_rate

        g = 0.5 * fcff_cagr + 0.5 * g_roic
        g = float(np.clip(g, 0.0, self.gdp_growth))

        if self.verbose:
            log(self.ticker, f"g_terminal={g:.4f}  FCFF_CAGR={fcff_cagr:.4f}  g_roic={g_roic:.4f}")
        return g

    # ─────────────────────────────────────────────────────────────
    # 6. Valuation
    # ─────────────────────────────────────────────────────────────

    def compute_valuation(self) -> "DCFModel":
        """DCF: EV = Σ FCFF/(1+r)^t + TV / (1+r)^T"""
        wacc = self.compute_wacc()
        df   = self.result_df.copy()

        # reinvestment_rate median (양수 NOPAT 기준)
        pos = df[df["nopat"] > 0]["reinvestment_rate"].dropna()
        reinv_med = float(self._winsorize(pos).median()) if not pos.empty else 0.3

        g = self.compute_terminal_growth(reinv_med, wacc)

        # TV 분모 안정성 체크
        if wacc - g < 0.01:
            g = wacc - 0.01
            if self.verbose: log(self.ticker, f"WACC-g 너무 작음 → g를 {g:.4f}로 조정")

        fcff_arr = df["fcff"].values
        T        = len(fcff_arr)

        # PV(FCFF)
        pv_fcff = sum(fcff_arr[t] / (1 + wacc) ** (t + 1) for t in range(T))

        # Terminal Value
        fcff_last = float(fcff_arr[-1])
        tv  = fcff_last * (1 + g) / (wacc - g)
        pv_tv = tv / (1 + wacc) ** T

        ev = pv_fcff + pv_tv

        # Net Debt = Total Debt - Cash
        net_debt = 0.0
        if not self._bs.empty:
            latest = self._bs.sort_values("date").iloc[-1]
            td   = pd.to_numeric(latest.get("totalDebt", 0), errors="coerce") or 0
            cash = pd.to_numeric(latest.get("cashAndCashEquivalents", 0), errors="coerce") or 0
            net_debt = float(td - cash)

        equity_val = ev - net_debt

        # 주식수 (diluted 우선 → basic)
        shares = np.nan
        if "weightedAverageShsOutDil" in self._inc.columns:
            s = pd.to_numeric(self._inc["weightedAverageShsOutDil"].iloc[-1], errors="coerce")
            if pd.notna(s) and s > 0:
                shares = float(s)
        if np.isnan(shares) and "weightedAverageShsOut" in self._inc.columns:
            s = pd.to_numeric(self._inc["weightedAverageShsOut"].iloc[-1], errors="coerce")
            if pd.notna(s) and s > 0:
                shares = float(s)

        target_price = equity_val / shares if (not np.isnan(shares) and shares > 0) else np.nan

        # 현재 주가
        current_price = np.nan
        try:
            r = requests.get(f"{FMP_BASE}/quote/{self.ticker}",
                              params={"apikey": self.api_key}, timeout=10)
            data = r.json()
            if isinstance(data, list) and data:
                current_price = float(data[0].get("price", np.nan) or np.nan)
        except Exception:
            pass

        upside = ((target_price / current_price) - 1) * 100 if (
            not np.isnan(target_price) and not np.isnan(current_price) and current_price > 0
        ) else np.nan

        self.valuation = {
            "ticker":           self.ticker,
            "wacc":             wacc,
            "g_terminal":       g,
            "reinvestment_rate":reinv_med,
            "pv_fcff":          pv_fcff,
            "terminal_value":   tv,
            "enterprise_value": ev,
            "net_debt":         net_debt,
            "equity_value":     equity_val,
            "shares":           shares,
            "target_price":     target_price,
            "current_price":    current_price,
            "upside_pct":       upside,
        }
        if self.verbose:
            log(self.ticker, f"EV={ev/1e9:.2f}B  TP={target_price:.2f}  Upside={upside:.1f}%")
        return self

    # ─────────────────────────────────────────────────────────────
    # 7. DB 저장
    # ─────────────────────────────────────────────────────────────

    def save_to_db(self, run_date: str = None) -> int:
        """result_df + valuation → us_fcff_dcf_valuation 저장"""
        if self.result_df is None or self.valuation is None:
            return 0
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v = self.valuation
        df = self.result_df.copy()

        rows = []
        for _, row in df.iterrows():
            rows.append({
                "date":             run_date,
                "ticker":           self.ticker,
                "quarter":          row.get("quarter"),
                "sales_forecast":   row.get("sales_forecast"),
                "opm_forecast":     row.get("opm_forecast"),
                "ebit":             row.get("ebit"),
                "tax_rate":         row.get("tax_rate"),
                "nopat":            row.get("nopat"),
                "da":               row.get("da"),
                "capex":            row.get("capex"),
                "nwc":              row.get("nwc"),
                "delta_nwc":        row.get("delta_nwc"),
                "fcff":             row.get("fcff"),
                "roic":             row.get("roic"),
                "reinvestment_rate":v["reinvestment_rate"],
                "g_terminal":       v["g_terminal"],
                "discount_rate":    v["wacc"],
                "enterprise_value": v["enterprise_value"],
                "net_debt":         v["net_debt"],
                "equity_value":     v["equity_value"],
                "shares":           v["shares"],
                "target_price":     v["target_price"],
                "current_price":    v["current_price"],
                "upside_pct":       v["upside_pct"],
            })

        save_df = pd.DataFrame(rows)
        # NaN → None 변환
        save_df = save_df.where(pd.notnull(save_df), None)

        sql = f"""
            INSERT INTO `{TABLE_RESULT}`
            (date, ticker, quarter, sales_forecast, opm_forecast, ebit, tax_rate,
             nopat, da, capex, nwc, delta_nwc, fcff, roic, reinvestment_rate,
             g_terminal, discount_rate, enterprise_value, net_debt, equity_value,
             shares, target_price, current_price, upside_pct)
            VALUES
            (%(date)s, %(ticker)s, %(quarter)s, %(sales_forecast)s, %(opm_forecast)s,
             %(ebit)s, %(tax_rate)s, %(nopat)s, %(da)s, %(capex)s, %(nwc)s,
             %(delta_nwc)s, %(fcff)s, %(roic)s, %(reinvestment_rate)s,
             %(g_terminal)s, %(discount_rate)s, %(enterprise_value)s, %(net_debt)s,
             %(equity_value)s, %(shares)s, %(target_price)s, %(current_price)s,
             %(upside_pct)s)
            ON DUPLICATE KEY UPDATE
                fcff=VALUES(fcff), target_price=VALUES(target_price),
                upside_pct=VALUES(upside_pct), enterprise_value=VALUES(enterprise_value)
        """
        conn = get_conn()
        try:
            with conn.cursor() as cur:
                cur.executemany(sql, save_df.to_dict("records"))
            conn.commit()
        except Exception:
            conn.rollback(); raise
        finally:
            conn.close()
        return len(rows)

    # ─────────────────────────────────────────────────────────────
    # 8. 시각화
    # ─────────────────────────────────────────────────────────────

    def plot(self) -> None:
        """FCFF 흐름 막대그래프 + 밸류에이션 요약"""
        if self.result_df is None:
            print("[WARN] result_df 없음 — compute_fcff() 먼저 실행하세요")
            return

        df = self.result_df.copy()
        v  = self.valuation or {}

        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        fig.suptitle(f"{self.ticker}  FCFF DCF Valuation", fontsize=14, fontweight="bold")

        # ① FCFF 막대
        ax = axes[0, 0]
        colors = ["#2ecc71" if f >= 0 else "#e74c3c" for f in df["fcff"]]
        ax.bar(df["quarter"], df["fcff"] / 1e9, color=colors, edgecolor="white")
        ax.axhline(0, color="black", lw=0.8)
        ax.set_title("Forecast FCFF (B$)"); ax.set_ylabel("USD Billion")
        ax.tick_params(axis="x", rotation=45); ax.grid(axis="y", alpha=0.3)

        # ② Sales & EBIT
        ax = axes[0, 1]
        ax.bar(df["quarter"], df["sales_forecast"] / 1e9,
               alpha=0.5, label="Sales", color="#3498db")
        ax.bar(df["quarter"], df["ebit"] / 1e9,
               alpha=0.7, label="EBIT", color="#e67e22")
        ax.set_title("Sales & EBIT (B$)"); ax.set_ylabel("USD Billion")
        ax.legend(fontsize=8); ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.3)

        # ③ OPM 추세
        ax = axes[1, 0]
        ax.plot(df["quarter"], df["opm_forecast"] * 100,
                marker="o", color="purple", lw=1.5)
        ax.axhline(0, color="black", lw=0.8)
        ax.set_title("Forecast OPM (%)"); ax.set_ylabel("%")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.1f}%"))
        ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)

        # ④ 밸류에이션 텍스트 요약
        ax = axes[1, 1]
        ax.axis("off")
        tp    = v.get("target_price", np.nan)
        cp    = v.get("current_price", np.nan)
        upsd  = v.get("upside_pct", np.nan)
        ev    = v.get("enterprise_value", np.nan)
        wacc  = v.get("wacc", np.nan)
        g     = v.get("g_terminal", np.nan)
        nd    = v.get("net_debt", np.nan)
        eqv   = v.get("equity_value", np.nan)

        def _fmt(x, unit="B", d=2):
            if np.isnan(x): return "N/A"
            if unit == "B": return f"${x/1e9:,.{d}f}B"
            if unit == "%": return f"{x*100:.{d}f}%"
            return f"${x:,.{d}f}"

        lines = [
            f"Current Price :  {_fmt(cp,'$')}",
            f"Target Price  :  {_fmt(tp,'$')}",
            f"Upside        :  {upsd:.1f}%" if not np.isnan(upsd) else "Upside : N/A",
            "─" * 30,
            f"Enterprise Value: {_fmt(ev)}",
            f"Net Debt        : {_fmt(nd)}",
            f"Equity Value    : {_fmt(eqv)}",
            "─" * 30,
            f"WACC          :  {_fmt(wacc,'%')}",
            f"g_terminal    :  {_fmt(g,'%')}",
        ]
        ax.text(0.05, 0.95, "\n".join(lines), transform=ax.transAxes,
                fontsize=10, verticalalignment="top", fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa", alpha=0.8))

        plt.tight_layout()
        plt.show()

    # ─────────────────────────────────────────────────────────────
    # 9. 편의 메서드: 전체 실행
    # ─────────────────────────────────────────────────────────────

    def run(self) -> "DCFModel":
        """load → compute_fcff → compute_valuation 순서 실행"""
        self.load_sales()
        self.load_financials()
        self.compute_fcff()
        self.compute_valuation()
        return self


# ── 단일 티커 처리 래퍼 ──────────────────────────────────────────
def process_one_ticker(
    ticker: str,
    engine,
    verbose: bool = False,
    run_date: str = None,
    save_db: bool = True,
) -> Dict[str, Any]:
    """
    DCFModel 실행 래퍼.
    반환: {status, ticker, target_price, upside_pct, rows_saved, msg}
    """
    run_date = run_date or datetime.now().strftime("%Y-%m-%d")
    try:
        model = DCFModel(ticker=ticker, engine=engine, verbose=verbose)
        model.run()

        rows = model.save_to_db(run_date) if save_db else 0
        v    = model.valuation
        return {
            "status":       "ok",
            "ticker":       ticker,
            "target_price": v.get("target_price", np.nan),
            "upside_pct":   v.get("upside_pct", np.nan),
            "wacc":         v.get("wacc", np.nan),
            "g_terminal":   v.get("g_terminal", np.nan),
            "rows_saved":   rows,
            "msg":          f"TP={v.get('target_price', np.nan):.2f}",
        }
    except Exception as e:
        return {
            "status": "fail",
            "ticker": ticker,
            "target_price": np.nan,
            "upside_pct":   np.nan,
            "wacc":         np.nan,
            "g_terminal":   np.nan,
            "rows_saved":   0,
            "msg":          str(e)[:120],
        }
    finally:
        clear_memory()


print("[OK] DCFModel 클래스 & process_one_ticker 정의 완료")

# ── 단일 티커 테스트 ─────────────────────────────────────────────
TEST_TICKER = "AAPL"
print(f"\n[테스트] {TEST_TICKER} 단일 실행 ...")
_res = process_one_ticker(TEST_TICKER, engine, verbose=True)
print(f"  status={_res['status']}  msg={_res['msg']}")
if _res["status"] == "ok":
    _model = DCFModel(ticker=TEST_TICKER, engine=engine, verbose=True).run()
    _model.plot()


[OK] DCFModel 클래스 & process_one_ticker 정의 완료

[테스트] AAPL 단일 실행 ...
  status=fail  msg=[AAPL] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)


## Cell 5 · 배치 실행

> `TICKER_START / TICKER_END` 로 범위 조정, `SKIP_DONE=True` 로 재시작 지원

In [5]:
# ─────────────────────────────────────────────────────────────────
# Cell 5 · 배치 실행
# ─────────────────────────────────────────────────────────────────

RUN_TICKERS = US_TICKER_LIST[TICKER_START:TICKER_END]
total   = len(RUN_TICKERS)
run_date = datetime.now().strftime("%Y-%m-%d")

done_set = set()
if SKIP_DONE and os.path.exists(DONE_PATH):
    with open(DONE_PATH) as f:
        done_set = {l.strip() for l in f if l.strip()}

ok_cnt = skip_cnt = fail_cnt = 0
results = []
t0 = time.time()

log("BATCH", f"배치 시작: {total:,}개  run_date={run_date}  SKIP_DONE={SKIP_DONE}")
print("=" * 70)

for idx, ticker in enumerate(RUN_TICKERS, 1):
    pct    = idx / total * 100
    prefix = f"[{idx:>5}/{total}] ({pct:5.1f}%) {ticker:<8}"

    if SKIP_DONE and ticker in done_set:
        print(f"{prefix} SKIP (checkpoint)", flush=True)
        skip_cnt += 1
        continue

    res = process_one_ticker(ticker, engine, verbose=False,
                              run_date=run_date, save_db=True)
    results.append(res)

    if res["status"] == "ok":
        tp  = res["target_price"]
        up  = res["upside_pct"]
        w   = res["wacc"]
        tp_str = f"TP=${tp:.2f}" if not np.isnan(tp) else "TP=N/A"
        up_str = f"↑{up:.1f}%" if not np.isnan(up) else ""
        print(f"{prefix} OK  {tp_str} {up_str}  WACC={w:.3f}", flush=True)
        with open(DONE_PATH, "a") as f:
            f.write(ticker + "\n")
        ok_cnt += 1
    else:
        print(f"{prefix} FAIL  {res['msg']}", flush=True)
        with open(FAIL_PATH, "a") as f:
            f.write(ticker + "\n")
        fail_cnt += 1

elapsed = time.time() - t0
print("=" * 70)
log("BATCH", f"완료  OK={ok_cnt}  SKIP={skip_cnt}  FAIL={fail_cnt}  "
             f"경과={elapsed:.0f}s  평균={elapsed/max(ok_cnt+fail_cnt,1):.1f}s/ticker")

# 결과 요약
if results:
    summary = pd.DataFrame(results)
    summary = summary[summary["status"]=="ok"].sort_values("upside_pct", ascending=False)
    print("\n[상위 업사이드 종목 TOP 20]")
    display(summary[["ticker","target_price","upside_pct","wacc","g_terminal"]].head(20))


[23:14:04][BATCH] 배치 시작: 10개  run_date=2026-04-09  SKIP_DONE=False
[    1/10] ( 10.0%) NVDA     FAIL  [NVDA] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[    2/10] ( 20.0%) GOOG     FAIL  [GOOG] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[    3/10] ( 30.0%) AAPL     FAIL  [AAPL] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[    4/10] ( 40.0%) MSFT     FAIL  [MSFT] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[    5/10] ( 50.0%) AMZN     FAIL  [AMZN] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[    6/10] ( 60.0%) TSM      FAIL  [TSM] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[    7/10] ( 70.0%) META     FAIL  [META] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[    8/10] ( 80.0%) AVGO     FAIL  [AVGO] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[    9/10] ( 90.0%) TSLA     FAIL  [TSLA] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[   10/10] (100.0%) LLY      FAIL  [LLY] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
[23:14:05][BATCH] 완료  OK=0  SKIP=0  FAIL=10  경과=1s  평균=

,ticker,target_price,upside_pct,wacc,g_terminal


## Cell 6 · 결과 조회 & 시각화

> `VIZ_TICKERS` 를 원하는 종목으로 변경

In [6]:
# ─────────────────────────────────────────────────────────────────
# Cell 6 · 결과 조회 & 시각화
# ─────────────────────────────────────────────────────────────────

# ── 6-1. DB 현황 ─────────────────────────────────────────────────
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT DATE(date) AS run_date,
                   COUNT(DISTINCT ticker) AS tickers,
                   COUNT(*) AS total_rows,
                   AVG(target_price) AS avg_tp,
                   AVG(upside_pct) AS avg_upside
            FROM {TABLE_RESULT}
            GROUP BY DATE(date)
            ORDER BY run_date DESC
            LIMIT 10
        """)
        _summary = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 60)
print(f"[DB 현황] {TABLE_RESULT}")
print("=" * 60)
display(_summary)

# ── 6-2. 최신 run_date 기준 업사이드 상위/하위 ────────────────────
conn = get_conn()
try:
    with conn.cursor() as cur:
        # 최신 run_date 찾기
        cur.execute(f"SELECT MAX(date) AS max_date FROM {TABLE_RESULT}")
        max_date = cur.fetchone()["max_date"]

        if max_date:
            cur.execute(f"""
                SELECT ticker, MAX(target_price) AS target_price,
                       MAX(current_price) AS current_price,
                       MAX(upside_pct) AS upside_pct,
                       MAX(discount_rate) AS wacc,
                       MAX(g_terminal) AS g_terminal
                FROM {TABLE_RESULT}
                WHERE date = %s AND target_price IS NOT NULL
                GROUP BY ticker
                ORDER BY upside_pct DESC
            """, (max_date,))
            results_df = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

if max_date and not results_df.empty:
    print(f"\n[최신 run_date: {max_date}]  총 {len(results_df)}개 종목")

    print("\n🔼 업사이드 TOP 20")
    display(results_df.head(20)[["ticker","target_price","current_price","upside_pct","wacc","g_terminal"]])

    print("\n🔽 다운사이드 TOP 20")
    display(results_df.tail(20)[["ticker","target_price","current_price","upside_pct","wacc","g_terminal"]])

    # 업사이드 분포 히스토그램
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Valuation Summary ({max_date})", fontsize=13)

    ax = axes[0]
    upside_clean = results_df["upside_pct"].dropna().clip(-200, 200)
    ax.hist(upside_clean, bins=40, color="#3498db", edgecolor="white", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.axvline(20, color="green", lw=1, ls="--", label="+20% (Buy zone)")
    ax.set_title("Upside Distribution"); ax.set_xlabel("Upside (%)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    grade_counts = pd.cut(results_df["upside_pct"],
                          bins=[-np.inf, -20, 0, 20, 50, np.inf],
                          labels=["Strong Sell (<-20%)", "Sell (-20~0%)",
                                  "Hold (0~20%)", "Buy (20~50%)", "Strong Buy (>50%)"])
    grade_counts.value_counts().sort_index().plot(
        kind="barh", ax=ax,
        color=["#c0392b","#e74c3c","#f39c12","#2ecc71","#27ae60"]
    )
    ax.set_title("Valuation Grade Distribution"); ax.grid(axis="x", alpha=0.3)

    plt.tight_layout(); plt.show()
else:
    print("[INFO] 데이터 없음 — Cell 5 배치 실행 후 재시도")

# ── 6-3. 개별 종목 시각화 ────────────────────────────────────────
VIZ_TICKERS = ["AAPL", "MSFT", "NVDA"]   # ← 원하는 종목으로 변경

print(f"\n[개별 종목 시각화] {VIZ_TICKERS}")
for tk in VIZ_TICKERS:
    try:
        m = DCFModel(ticker=tk, engine=engine, verbose=False).run()
        m.plot()
    except Exception as e:
        print(f"  [{tk}] 시각화 실패: {e}")


[DB 현황] us_fcff_dcf_valuation


""


[INFO] 데이터 없음 — Cell 5 배치 실행 후 재시도

[개별 종목 시각화] ['AAPL', 'MSFT', 'NVDA']
  [AAPL] 시각화 실패: [AAPL] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
  [MSFT] 시각화 실패: [MSFT] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
  [NVDA] 시각화 실패: [NVDA] 예측 Sales 없음 (Ensemble/SARIMA/ETS/Theta 모두 없음)
